## Summary and Next Steps

**Key Results:**

1. **Exact benchmark:** Verified TFIM phase transition near $g/J = 1$
2. **RBM expressivity:** Showed convergence with hidden units
3. **Order parameters:** Connected RBM weights to statistical-mechanics phases

**For exam write-up:**

1. **Include plots:** Phase diagram, convergence curves, order parameters vs. g
2. **Discuss limitations:** Monte Carlo sampling noise, finite-system effects
3. **Link to theory:** Mean-field predictions, replica analysis, entropy crisis
4. **Interpret physics:** Glassy dynamics near criticality, feature organization in RBM

This project demonstrates that neural quantum states provide a bridge between quantum many-body physics, statistical mechanics, and machine learning.


## Connection to Exam Topics: Physical Interpretation

### 1. Path Integrals and Variational Projection

The variational energy functional:
$$E(\theta) = \frac{\langle \Psi(\theta) | H | \Psi(\theta) \rangle}{\langle \Psi(\theta) | \Psi(\theta) \rangle}$$

is an upper bound on the true ground state energy. This relates to **imaginary-time evolution**:

- The Euclidean path integral projects onto the ground state: $\mathcal{D}[\psi] e^{-S_E[\psi]}$
- The RBM manifold restricts this path integral to a subspace
- Minimizing $E(\theta)$ is equivalent to **variational projection** onto the NQS manifold

### 2. Stochastic Dynamics & Molecular Dynamics

The parameter updates via natural gradient descent:
$$\dot{\theta} = -\eta \nabla_\theta E$$

can be interpreted as **stochastic dynamics** on the parameter landscape:

- With MC noise: $d\theta \sim -\eta \nabla_\theta E \, dt + \sqrt{\eta/N} \, dW_t$
- Analogous to **Langevin dynamics** in statistical mechanics
- Near phase transitions: dynamics slow down (critical slowing down)

### 3. Statistical Mechanics of RBMs

The RBM energy:
$$E(\sigma,h) = -\sum_i a_i\sigma_i - \sum_j b_j h_j - \sum_{ij} W_{ij}\sigma_i h_j$$

maps to a **spin-glass model**:

- **Disordered regime** ($g > 1$): RBM exhibits glassy structure
  - Multiple local minima in parameter space
  - Low sparsity (many active weights)
  - Slow optimization dynamics
- **Ordered regime** ($g < 1$): Simpler RBM structure
  - Fewer hidden feature groups
  - Higher sparsity in weights
  - Faster convergence

- **Critical region** ($g \approx 1$): Transition between regimes
  - Feature proliferation
  - Increased optimization difficulty
  - Diverging correlation lengths


In [ ]:
def order_parameters(theta, L, M):
    """Compute order parameters from trained RBM.
    
    Returns:
        dict with weight statistics
    """
    W = theta['W']
    
    op = {
        'mean_abs_W': np.mean(np.abs(W)),
        'std_W': np.std(W),
        'max_W': np.max(np.abs(W)),
        'sparsity': float(np.sum(np.abs(W) < 0.01) / W.size),
        'condition_number': np.linalg.cond(W),
    }
    
    # Singular value analysis
    U, s, Vh = np.linalg.svd(W, full_matrices=False)
    op['singular_values'] = s
    op['truncation_ratio'] = s[-1] / s[0]  # Ratio of smallest to largest SV
    
    return op

# Demo: compute order parameters for the trained model
theta_demo = history_demo['final_theta']
op_demo = order_parameters(theta_demo, L_demo, M_demo)

print(f"\nOrder Parameters for trained RBM (L={L_demo}, M={M_demo}, g={g_demo}):")
print(f"  Mean |W_{ij}|:     {op_demo['mean_abs_W']:.6f}")
print(f"  Std W:            {op_demo['std_W']:.6f}")
print(f"  Max |W_{ij}|:      {op_demo['max_W']:.6f}")
print(f"  Sparsity (<0.01): {op_demo['sparsity']:.4f}")
print(f"  Condition number: {op_demo['condition_number']:.2e}")
print(f"  SV truncation:    {op_demo['truncation_ratio']:.4f}")

## RBM Order Parameters: Weight Statistics and Analysis

After training, extract order parameters from the RBM and relate them to statistical-mechanics phases.


In [ ]:
print("Training RBM NQS for L=8, g=1.0, M=8...")
print("-" * 60)

L_demo = 8
g_demo = 1.0
M_demo = 8
E0_exact_demo = exact_results[(L_demo, g_demo)]['E0']

history_demo = train_vmc(L_demo, M_demo, g_demo, J=J, 
                         n_sweeps=50, n_samples=300, lr=0.1, seed=42)

E_final_demo = history_demo['energies'][-1]
E_error_demo = E_final_demo - E0_exact_demo

print(f"\nResults:")
print(f"  Exact E0:      {E0_exact_demo:.6f}")
print(f"  VMC E (final): {E_final_demo:.6f}")
print(f"  Error ΔE:      {E_error_demo:.6f}")

# Plot convergence
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Energy convergence
axes[0, 0].plot(history_demo['energies'], 'b-', linewidth=2, label='E_RBM(t)')
axes[0, 0].axhline(E0_exact_demo, color='r', linestyle='--', label='E_0 (exact)', linewidth=2)
axes[0, 0].set_ylabel('Energy', fontsize=11)
axes[0, 0].set_title('Energy Convergence', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Energy variance
axes[0, 1].semilogy(history_demo['variances'], 'g-', linewidth=2)
axes[0, 1].set_ylabel('Variance', fontsize=11)
axes[0, 1].set_title('Energy Variance', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# Acceptance rate
axes[1, 0].plot(history_demo['accept_rates'], 'k-', linewidth=2)
axes[1, 0].set_ylabel('Acceptance Rate', fontsize=11)
axes[1, 0].set_xlabel('Optimization Step', fontsize=11)
axes[1, 0].set_title('Metropolis Acceptance Rate', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)

# Gradient norms
grad_norms = np.array(history_demo['grad_norms'])
axes[1, 1].semilogy(grad_norms[:, 0], label='∂E/∂a', linewidth=2)
axes[1, 1].semilogy(grad_norms[:, 1], label='∂E/∂b', linewidth=2)
axes[1, 1].semilogy(grad_norms[:, 2], label='∂E/∂W', linewidth=2)
axes[1, 1].set_ylabel('Gradient Norm', fontsize=11)
axes[1, 1].set_xlabel('Optimization Step', fontsize=11)
axes[1, 1].set_title('Parameter Gradient Norms', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/vmc_demo_convergence.png', dpi=fig_dpi)
plt.show()

## Demo: Single System L=8, g=1.0

Run one training loop to demonstrate convergence.


In [ ]:
def train_vmc(L, M, g, J=1.0, n_sweeps=100, n_samples=500, lr=0.1, seed=None):
    """Run VMC optimization for TFIM with RBM ansatz.
    
    Args:
        L: system size
        M: number of hidden units
        g: transverse field
        J: Ising coupling
        n_sweeps: number of optimization steps
        n_samples: MC samples per sweep
        lr: learning rate
        seed: random seed
    
    Returns:
        dict with training history and final parameters
    """
    if seed is not None:
        rng = np.random.RandomState(seed)
    else:
        rng = np.random.RandomState()
    
    # Initialize parameters
    theta = {
        'a': rng.randn(L) * 0.01,
        'b': rng.randn(M) * 0.01,
        'W': rng.randn(L, M) * 0.01,
    }
    
    history = {
        'energies': [],
        'variances': [],
        'accept_rates': [],
        'grad_norms': [],
    }
    
    for sweep in range(n_sweeps):
        # Sample configurations
        configs, acc_rate = metropolis_sampler(n_samples, 1, theta, L, rng=rng)
        
        # Compute local energies
        E_locs = np.array([local_energy(sigma, theta, L, J=J, g=g) for sigma in configs])
        E_mean = np.mean(E_locs)
        E_var = np.var(E_locs)
        
        # Compute energy gradients (natural gradient terms)
        grad_a = np.zeros(L)
        grad_b = np.zeros(M)
        grad_W = np.zeros((L, M))
        
        for i, sigma in enumerate(configs):
            force = E_locs[i] - E_mean  # Force for this sample
            grads = compute_gradients(sigma, theta)
            
            grad_a += force * grads['a']
            grad_b += force * grads['b']
            grad_W += force * grads['W']
        
        grad_a /= n_samples
        grad_b /= n_samples
        grad_W /= n_samples
        
        # Update parameters (gradient descent)
        theta['a'] -= lr * grad_a
        theta['b'] -= lr * grad_b
        theta['W'] -= lr * grad_W
        
        # Store history
        history['energies'].append(E_mean)
        history['variances'].append(E_var)
        history['accept_rates'].append(acc_rate)
        history['grad_norms'].append((np.linalg.norm(grad_a), np.linalg.norm(grad_b), np.linalg.norm(grad_W)))
        
        if (sweep + 1) % 20 == 0 or sweep == 0:
            print(f"  Step {sweep+1:3d}: E = {E_mean:8.4f} ± {np.sqrt(E_var/n_samples):6.4f}, acc_rate = {acc_rate:.3f}")
    
    history['final_theta'] = theta
    
    return history

print("VMC training function defined.")

## VMC Training Loop

Implement the full variational Monte Carlo optimization loop.


In [ ]:
def local_energy(sigma, theta, L, J=1.0, g=1.0):
    """Compute local energy E_loc(σ) = <σ|H|Ψ> / <σ|Ψ> for TFIM.
    
    H = -J Σ_i σ_z_i σ_z_{i+1} - g Σ_i σ_x_i
    
    Args:
        sigma: spin configuration
        theta: RBM parameters
        L: system size
        J: Ising coupling
        g: transverse field
    
    Returns:
        E_loc (scalar)
    """
    E_loc = 0.0
    
    # ZZ diagonal terms
    for i in range(L):
        i_next = (i + 1) % L
        E_loc += -J * sigma[i] * sigma[i_next]
    
    # X off-diagonal terms (flip each spin)
    for i in range(L):
        sigma_flip = sigma.copy()
        sigma_flip[i] *= -1
        
        # Ratio: <σ_flip|Ψ> / <σ|Ψ>
        psi_ratio = psi_squared_ratio(sigma, sigma_flip, theta)
        E_loc += -g * np.sqrt(psi_ratio)
    
    return E_loc

print("Local energy function defined.")

## Local Energy Estimator for TFIM

Compute E_loc(σ) for each configuration by evaluating the Hamiltonian matrix element.


In [ ]:
def metropolis_sampler(n_samples, n_steps, theta, L, rng=None):
    """Metropolis-Hastings sampling from |Ψ(θ)|^2.
    
    Args:
        n_samples: Number of samples to generate
        n_steps: Metropolis steps between samples
        theta: RBM parameters
        L: System size
        rng: Random number generator
    
    Returns:
        configs: Array of shape (n_samples, L)
        accept_rate: Fraction of accepted moves
    """
    if rng is None:
        rng = np.random.RandomState()
    
    configs = np.zeros((n_samples, L), dtype=np.int8)
    
    # Initial configuration
    sigma = rng.choice([-1, 1], size=L)
    
    # Burn-in
    n_accepted = 0
    for _ in range(50):
        sigma_new = sigma.copy()
        i = rng.randint(0, L)
        sigma_new[i] *= -1
        
        accept_prob = psi_squared_ratio(sigma, sigma_new, theta)
        if rng.rand() < accept_prob:
            sigma = sigma_new
            n_accepted += 1
    
    # Sampling
    for sample_idx in range(n_samples):
        for step in range(n_steps):
            sigma_new = sigma.copy()
            i = rng.randint(0, L)
            sigma_new[i] *= -1
            
            accept_prob = psi_squared_ratio(sigma, sigma_new, theta)
            if rng.rand() < accept_prob:
                sigma = sigma_new
                n_accepted += 1
        
        configs[sample_idx] = sigma
    
    accept_rate = n_accepted / (50 + n_samples * n_steps)
    return configs, accept_rate

print("Metropolis sampler defined.")

## Metropolis Sampling

Generate sample configurations from |Ψ(θ)|² using Metropolis-Hastings algorithm.


In [ ]:
# Test normalization for small L
def generate_all_configs(L):
    """Generate all 2^L spin configurations."""
    n_configs = 2 ** L
    configs = np.zeros((n_configs, L), dtype=np.int8)
    for i in range(n_configs):
        binary = format(i, f'0{L}b')
        configs[i] = np.array([1 if b == '0' else -1 for b in binary])
    return configs

# Initialize RBM for L=4 to test
L_test = 4
M_test = 4
theta_test = {
    'a': np.random.randn(L_test) * 0.1,
    'b': np.random.randn(M_test) * 0.1,
    'W': np.random.randn(L_test, M_test) * 0.1,
}

configs_test = generate_all_configs(L_test)
log_psis = np.array([log_psi(sigma, theta_test) for sigma in configs_test])
psi_squared = np.exp(2 * log_psis)
psi_squared_normalized = psi_squared / np.sum(psi_squared)

print(f"\nNormalization test for L={L_test}, M={M_test}:")
print(f"  Sum of |Ψ|^2: {np.sum(psi_squared):.6f}")
print(f"  Sum of normalized |Ψ|^2: {np.sum(psi_squared_normalized):.6f} (should be 1)")
print(f"  Max amplitude: {np.max(psi_squared_normalized):.6f}")

In [ ]:
def log_psi(sigma, theta):
    """Compute log|Psi_RBM(sigma)| for a spin configuration.
    
    Ψ_RBM(σ) ∝ exp(Σ_i a_i σ_i) ∏_j 2 cosh(b_j + Σ_i W_ij σ_i)
    
    Args:
        sigma: spin configuration (L,) with values in {-1, +1}
        theta: dict with keys 'a', 'b', 'W'
    
    Returns:
        log|Ψ|
    """
    a, b, W = theta['a'], theta['b'], theta['W']
    
    # First term: sum_i a_i sigma_i
    term1 = np.dot(a, sigma)
    
    # Second term: sum_j log(2*cosh(b_j + sum_i W_ij sigma_i))
    h_fields = b + np.dot(W.T, sigma)  # Shape (M,)
    term2 = np.sum(np.log(2.0 * np.cosh(h_fields) + 1e-10))
    
    return term1 + term2

def psi_squared_ratio(sigma_old, sigma_new, theta):
    """Compute |Ψ(σ_new)|^2 / |Ψ(σ_old)|^2 efficiently for Metropolis sampling."""
    log_old = log_psi(sigma_old, theta)
    log_new = log_psi(sigma_new, theta)
    return np.exp(2.0 * (log_new - log_old))

def compute_gradients(sigma, theta):
    """Compute ∂log|Ψ|/∂θ for parameter updates.
    
    Returns:
        Dict with gradients for 'a', 'b', 'W'
    """
    a, b, W = theta['a'], theta['b'], theta['W']
    
    h_fields = b + np.dot(W.T, sigma)
    tanh_h = np.tanh(h_fields)
    
    grads = {
        'a': sigma.copy(),
        'b': tanh_h.copy(),
        'W': np.outer(sigma, tanh_h),  # Shape (L, M)
    }
    
    return grads

print("RBM amplitude functions defined.")

## RBM Ansatz: Parameters and Log-Amplitude

Define the RBM wave function and verify normalization for small systems.


In [ ]:
# Plot phase transition signature: energy per spin vs. g
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

for L in L_values:
    g_vals = [g for (L_val, g) in exact_results.keys() if L_val == L]
    e_per_spins = [exact_results[(L, g)]['e_per_spin'] for g in g_vals]
    
    ax.plot(g_vals, e_per_spins, 'o-', label=f'L={L}', markersize=6, linewidth=2)

ax.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='Critical g/J=1')
ax.set_xlabel('Transverse field g/J', fontsize=12)
ax.set_ylabel('Ground state energy per spin', fontsize=12)
ax.set_title('TFIM Phase Transition Signature', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/exact_phase_diagram.png', dpi=fig_dpi)
plt.show()

print("Phase diagram plotted.")

In [ ]:
def pauli_x():
    return np.array([[0, 1], [1, 0]], dtype=np.float64)

def pauli_z():
    return np.array([[1, 0], [0, -1]], dtype=np.float64)

def pauli_i():
    return np.eye(2, dtype=np.float64)

def build_tfim_hamiltonian(L, J=1.0, g=1.0):
    """Build TFIM Hamiltonian: H = -J sum_i sigma_z_i sigma_z_{i+1} - g sum_i sigma_x_i"""
    Ham = None
    
    # ZZ coupling terms
    for i in range(L):
        i_next = (i + 1) % L
        ops = []
        for j in range(L):
            if j == i or j == i_next:
                ops.append(pauli_z())
            else:
                ops.append(pauli_i())
        
        term = ops[0]
        for op in ops[1:]:
            term = np.kron(term, op)
        
        if Ham is None:
            Ham = -J * term
        else:
            Ham = Ham - J * term
    
    # X field terms
    for i in range(L):
        ops = []
        for j in range(L):
            if j == i:
                ops.append(pauli_x())
            else:
                ops.append(pauli_i())
        
        term = ops[0]
        for op in ops[1:]:
            term = np.kron(term, op)
        
        Ham = Ham - g * term
    
    return Ham

# Compute exact ground state energies
exact_results = {}

for L in L_values:
    for g in g_values:
        H = build_tfim_hamiltonian(L, J=J, g=g)
        eigenvalues = np.linalg.eigvalsh(H)
        E0 = eigenvalues[0]
        e_per_spin = E0 / L
        
        exact_results[(L, g)] = {
            'E0': E0,
            'e_per_spin': e_per_spin
        }
        
        print(f"L={L:2d}, g={g:.1f}: E0 = {E0:8.4f}, e/L = {e_per_spin:8.4f}")

print("\nExact ground state energies computed.")

## Exact Diagonalization of 1D TFIM

Build the Hamiltonian using Kronecker products and diagonalize to get exact ground state energies.


In [ ]:
# System parameters
L_values = [8, 10, 12]           # System sizes
J = 1.0                          # Ising coupling
g_values = [0.5, 1.0, 1.5, 2.0]  # Transverse field values

# RBM and VMC parameters
hidden_ratios = [0.5, 1.0, 2.0]  # M/L ratios for hidden units
N_SAMPLES = 500                   # MC samples per iteration
N_SWEEPS = 100                    # Optimization steps
LEARNING_RATE = 0.1              # SGD learning rate
MC_STEPS = 50                     # Metropolis steps between samples

print("Configuration:")
print(f"  System sizes L: {L_values}")
print(f"  Transverse fields g: {g_values}")
print(f"  Hidden unit ratios M/L: {hidden_ratios}")
print(f"  VMC samples per sweep: {N_SAMPLES}")
print(f"  Optimization sweeps: {N_SWEEPS}")

## Configuration

Define hyperparameters for the experiment.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh
from scipy.sparse import kron, eye, diags
import json
from pathlib import Path
import sys

# Add src directory to path
sys.path.insert(0, str(Path('.').parent / 'src'))

# Set random seed for reproducibility
np.random.seed(42)

# Display settings
plt.style.use('seaborn-v0_8-darkgrid')
fig_dpi = 100

# Neural Quantum State (RBM) for the 1D Transverse-Field Ising Model

**Exam Project: Statistical Mechanics, Path Integrals, and Neural Quantum States**

This notebook implements and analyzes a Restricted Boltzmann Machine (RBM) as a neural quantum state (NQS) approximating the ground state of the 1D transverse-field Ising model (TFIM). We link the numerical study to three exam topics:

1. **Path integrals:** imaginary-time evolution and variational projection
2. **Molecular/stochastic dynamics:** parameter updates on the NQS manifold
3. **Statistical mechanics of RBMs:** phase diagrams and compositional phases
